# Langchain: The basics

#### Developed By: Manaranjan Pradhan
#### www.manaranjanp.com

*This Jupyter notebook is confidential and proprietary to Manaranjan Pradhan. It is intended solely for authorized training purposes. Unauthorized distribution, sharing, or reproduction of this notebook or its contents is strictly prohibited. This material is for personal learning within the training program only and may not be used for commercial purposes or shared with others. Unauthorized use may result in disciplinary action or legal consequences. If you have received this notebook without authorization, please contact manaranjan@gmail.com immediately and delete all copies.*

In [6]:
!pip -q install langchain langchain-groq langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.6 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

In [3]:
#os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY')
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


## Zero Shot Prompting



In [7]:
from langchain_groq import ChatGroq
from langchain_classic.chains import LLMChain

In [8]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=256,
    max_retries=2,
)

In [20]:
text = """What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
Sentiment:"""

print(llm.invoke(text))

content='Sentiment: Positive. The customer uses words like "fresh", "flavorful", and "loved" to describe their experience, indicating a very positive sentiment.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 80, 'total_tokens': 115, 'completion_time': 0.091873476, 'completion_tokens_details': None, 'prompt_time': 0.007129738, 'prompt_tokens_details': None, 'queue_time': 0.069749221, 'total_time': 0.099003214}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019c6ee9-1f5f-7b63-94fa-1b5c635118b2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 35, 'total_tokens': 115}


## Classifying List of Reviews

In [21]:
reviews = [
    "The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.",
    "Impressive service! The staff was attentive and made excellent recommendations. Thoroughly enjoyed the evening.",
    "The restaurant's interior was a visual treat, beautifully paired with their gourmet dishes. A delightful experience!",
    "The food tasted alright, but the tables were not very clean, which was off-putting.",
    "Waited 30 minutes even with a reservation, and the main course was served cold. Disappointing visit."
]

## Using Prompt Templates

In [14]:
from langchain_classic import PromptTemplate

In [15]:
sentiment_template = """
What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: {review_text}
Sentiment:
"""

review_prompt = PromptTemplate(
    input_variables=["review_text"],
    template=sentiment_template,
)

### Generating Prompt with Templates

In [22]:
print(review_prompt.format(review_text=reviews[0]))


What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
Sentiment:



### Calling LLM with the prompt template

In [17]:
sentiment_chain = review_prompt | llm

In [18]:
response = sentiment_chain.invoke({"review_text": reviews[0]})

response.content

'Sentiment: Positive. The customer uses words like "fresh", "flavorful", "loved", and mentions the "chic décor" and "ambiance", indicating a very positive experience.'

In [19]:
print("================================")

for review in reviews:
    print(review_prompt.format(review_text=review))
    response = sentiment_chain.invoke({"review_text": review})
    print(response.content)
    print("================================")


What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
Sentiment:

Sentiment: Positive. The customer uses words like "fresh", "flavorful", and "loved" to describe their experience, indicating a positive opinion of the restaurant.

What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: Impressive service! The staff was attentive and made excellent recommendations. Thoroughly enjoyed the evening.
Sentiment:

Sentiment: Positive

What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: The restaurant's interior was a visual treat, beautifully paired with their gourmet dishes. A delightful experience!
Sentiment:

Sentiment: Positive

What is the sentiment of the customer review given below? It should be a positive or nega

## Classifying Categories

In [23]:
category_template = """
Classify the reviews into one the four categories as given in the examples.

review: The grilled chicken was seasoned to perfection and simply melted in the mouth.
category: Food Quality

review: Despite the crowd, the place was immaculately clean and the restrooms were spotless.
category: Overall Hygiene

review: The dim lighting and soothing jazz music provided an intimate and romantic setting.
category: Restaurant Ambience

review: We were kept waiting for our table even after a confirmed reservation and the staff seemed disinterested.
category: Customer Service

review: {review_text}
category:
"""

category_prompt = PromptTemplate(
    input_variables=["review_text"],
    template=category_template,
)

In [24]:
# We can now generate a prompt using the `format` method.
print(category_prompt.format(review_text=reviews[0]))


Classify the reviews into one the four categories as given in the examples.

review: The grilled chicken was seasoned to perfection and simply melted in the mouth.
category: Food Quality

review: Despite the crowd, the place was immaculately clean and the restrooms were spotless.
category: Overall Hygiene

review: The dim lighting and soothing jazz music provided an intimate and romantic setting.
category: Restaurant Ambience

review: We were kept waiting for our table even after a confirmed reservation and the staff seemed disinterested.
category: Customer Service

review: The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
category:



In [25]:
category_chain = category_prompt | llm

In [26]:
# Run the chain only specifying the input variable.
response = category_chain.invoke({"review_text": reviews[0]})

response.content

'Based on the given examples, I would categorize the review as:\n\ncategory: Food Quality and Restaurant Ambience\n\nHowever, since the review mentions both food quality and ambiance, it can be classified under two categories. But if I had to choose one, I would say:\n\ncategory: Food Quality\n\nThe review primarily talks about the seafood platter being "fresh and flavorful", which suggests that the food quality is the main focus. The mention of décor and ambiance is secondary.'

In [27]:
for review in reviews:
    print(review)
    print(category_chain.invoke({"review_text": review}))
    print("================================")

The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
content='Based on the given examples, I would categorize the review as:\n\ncategory: Food Quality and Restaurant Ambience\n\nHowever, since the review mentions both food quality and ambiance, it can\'t be classified into a single category as per the given examples. But if I had to choose one, I would say:\n\ncategory: Food Quality \n\n(The review mentions "The seafood platter was fresh and flavorful" which directly relates to food quality, and the mention of ambiance is more of a secondary comment)' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 170, 'total_tokens': 270, 'completion_time': 0.383084859, 'completion_tokens_details': None, 'prompt_time': 0.025877771, 'prompt_tokens_details': None, 'queue_time': 0.080662685, 'total_time': 0.40896263}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_bebe2dd4fb', 'service_tier

## Chaining Multiple Prompts

In [28]:
response_to_customer = """
"Write an appropriate and concise response to the customer by either appreciating their positive experience or adressing the specific concern that customer might
have mentioned in the feedback below.

<feedback>
{feedback}
</feedback>

The feedback is about {category}.

The goal of the response is to ensure customer engagement.
"""

In [31]:
response_prompt = PromptTemplate(
    input_variables=["feedback", "category"],
    template=response_to_customer,
)

In [32]:
final_response_chain = response_prompt | llm

In [33]:
complete_chain = (
    {
        "feedback": sentiment_chain,
        "category": category_chain,
    }
    | final_response_chain
)

In [34]:
from pprint import pprint

In [35]:
reviews[3]

'The food tasted alright, but the tables were not very clean, which was off-putting.'

In [36]:
response = complete_chain.invoke({"review_text": reviews[3]})

In [37]:
pprint(response.content)

('Dear valued customer,\n'
 '\n'
 'Thank you for taking the time to share your feedback about your recent visit '
 'to our restaurant. We apologize for the unclean tables, which clearly had a '
 'negative impact on your experience. We understand that cleanliness is a top '
 'priority, and we fell short of our standards. We will take immediate action '
 'to ensure that our tables are always clean and well-maintained.\n'
 '\n'
 'Although you mentioned that the food "tasted alright," we regret that the '
 'off-putting condition of the tables overshadowed your dining experience. We '
 'appreciate your feedback and would like to invite you to give us another '
 "chance to serve you better. Please let us know if there's anything else we "
 'can do to make things right.\n'
 '\n'
 'Thank you for your feedback, and we look forward to serving you again in the '
 'future.\n'
 '\n'
 'Best regards,\n'
 '[Your Restaurant Name]')
